In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)
train.head()

Train Shape: (630000, 15)
Test Shape: (270000, 14)


,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [5]:
TARGET = "Heart Disease"

X = train.drop(TARGET, axis=1)
y = train[TARGET]

In [6]:
from sklearn.preprocessing import LabelEncoder

for col in X.columns:
    if X[col].dtype == "object":
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        test[col] = le.transform(test[col])


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"Training Fold {fold+1}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=8,
        random_state=42
    )

    model.fit(X_train, y_train)

    test_preds += model.predict_proba(test)[:, 1] / 5

Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5


In [8]:
final_pred = (test_preds > 0.5).astype(int)


In [9]:
sample_submission = pd.read_csv("sample_submission.csv")
submission = sample_submission.copy()
submission[TARGET] = final_pred

submission.to_csv("final_submission.csv", index=False)

submission.head()


,id,Heart Disease
0,630000,1
1,630001,0
2,630002,1
3,630003,0
4,630004,0
